# exp075_compact_tracker_pfbeam_feature_repro_guard train

Train LightGBM on the compact PF/Beam tracker feature CSV generated by the separate PF/Beam feature notebook. This notebook does not regenerate train features.

## Contents

1. Setup and configuration
2. Generated PF/Beam feature check
3. LightGBM training
4. Metrics, feature importance, and artifacts

## 1. Setup and configuration

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, display

from settings import ExperimentPaths, load_config, get_nested
from compact_tracker_pfbeam_repro_guard import (
    TRACKER_TRAIN_FEATURES,
    OUTPUT_PREFIX,
    find_artifact,
    run_reproducibility_guard,
)

def cfg_get(config, dotted_key, default=None):
    value = get_nested(config, dotted_key)
    return default if value is None else value


In [ ]:
paths = ExperimentPaths()
paths.ensure_output_dirs()
config = load_config()

print("Experiment:", config["experiment"]["name"])
print("Route:", config["experiment"]["route"])
print("Audit mode:", cfg_get(config, "audit.mode"))
print("Feature source:", cfg_get(config, "model.feature_source"))
print("Kernel sources:", cfg_get(config, "runtime.kaggle.kernel_sources"))
print("Active modes:", cfg_get(config, "model.training.active_modes"))


## 2. Generated PF/Beam feature check

In [ ]:
tracker_path = find_artifact(
    TRACKER_TRAIN_FEATURES,
    cfg_get(config, "data.generated_tracker_features_train_local"),
)
print("Generated tracker/PF/Beam train features:", tracker_path)
preview = pd.read_csv(tracker_path, nrows=5, dtype={"id": str, "well": str})
print("Columns:", len(preview.columns))
display(preview)


## 3. LightGBM training

In [ ]:
summary = run_reproducibility_guard(
    output_dir=paths.artifacts_dir,
    tracker_path=tracker_path,
    modes=cfg_get(config, "model.training.modes", {}),
    active_modes=cfg_get(config, "model.training.active_modes", []),
    n_splits=int(cfg_get(config, "validation.n_folds", 5)),
    fast=bool(cfg_get(config, "audit.fast", False)),
    early_stopping_rounds=int(cfg_get(config, "model.training.early_stopping_rounds", 250)),
    max_rows=cfg_get(config, "model.training.max_rows"),
    max_train_rows=cfg_get(config, "model.training.max_train_rows"),
    save_models=bool(cfg_get(config, "model.training.save_models", True)),
    save_predictions=bool(cfg_get(config, "model.training.save_predictions", True)),
)
print(json.dumps(summary, indent=2))


## 4. Metrics, feature importance, and artifacts

In [ ]:
metrics = pd.read_csv(paths.artifacts_dir / f"{OUTPUT_PREFIX}_metrics.csv")
by_well = pd.read_csv(paths.artifacts_dir / f"{OUTPUT_PREFIX}_by_well.csv")
schema = pd.read_csv(paths.artifacts_dir / f"{OUTPUT_PREFIX}_feature_schema.csv")
importance_fold_mean = pd.read_csv(paths.artifacts_dir / f"{OUTPUT_PREFIX}_feature_importance_fold_mean.csv")
importance_fold_plot = paths.artifacts_dir / f"{OUTPUT_PREFIX}_feature_importance_fold_mean_top.png"
manifest_path = paths.artifacts_dir / f"{OUTPUT_PREFIX}_lgb_models" / "manifest.json"

display(metrics.sort_values(["mode", "model", "fold"]).head(60))
display(metrics[metrics["fold"].astype(str).eq("pooled")].sort_values("rmse_tvt"))
display(importance_fold_mean.head(40))
if importance_fold_plot.exists():
    display(Image(filename=str(importance_fold_plot)))
display(by_well.head(30))
print("Feature count:", len(schema))
print("Model manifest:", manifest_path, "exists=", manifest_path.exists())
